# 12 — LC25000 subtype task data preparation

Same LC25000 lung subset as notebook 07, but the task is **lung_aca (1) vs lung_scc (0)** — two carcinoma subtypes. `lung_n` (benign) is excluded. This is a harder visual discrimination than malignant-vs-benign and is where we expect to see actual differences between models.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import yaml

from utils.seed import set_seed
from utils.data_histology import load_lc25000, subset_lung_subtype, stratified_subsample

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])
cfg

In [ ]:
df = load_lc25000(cfg['data']['root'])
print('total images:', len(df))
df['class'].value_counts()

In [ ]:
sub = subset_lung_subtype(df)
max_per_class = cfg['data'].get('max_per_class')
if max_per_class:
    sub = stratified_subsample(sub, per_class=max_per_class, seed=cfg['seed'])
    print(f'subsampled to {max_per_class}/class')

print('subtype subset:', len(sub))
print('label distribution (1=aca, 0=scc):')
print(sub['label_binary'].value_counts())
print('pos rate:', sub['label_binary'].mean())

In [ ]:
cache_dir = Path('..') / cfg['data']['cache_dir']
cache_dir.mkdir(parents=True, exist_ok=True)
sub.to_csv(cache_dir / 'labels.csv', index=False)
print('saved →', cache_dir / 'labels.csv')

In [ ]:
# Visual sanity check — aca vs scc side by side.
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, cls in enumerate(['lung_aca', 'lung_scc']):
    sample = sub[sub['class'] == cls].sample(4, random_state=cfg['seed'])
    for j, (_, row) in enumerate(sample.iterrows()):
        ax = axes[i, j]
        ax.imshow(Image.open(row['path']))
        ax.set_title(f"{row['class']} (y={row['label_binary']})"); ax.axis('off')
plt.tight_layout(); plt.show()